# Comprehensive Research Evaluation & Ablation Study (300 Utterances)
**Nepali-English Code-Mixed Whisper ASR**

This notebook replicates the core experiments for the ACL 2023 paper on the expanded **300-utterance test set**:
1. **Unconstrained vs Constrained Decoding Ablation**: Proves the byte-identical output claim.
2. **Zero-Shot English Forcing (Task 6)**: Analyzes Whisper's performance when forced to English.
3. **Table 4 Generation**: Computes Overall WER, CER, Nep-WER, and CM-WER for the fine-tuned model and zero-shot baseline.


In [ ]:
!pip install -q jiwer peft torchao soundfile


In [ ]:
import os
import torch
import pandas as pd
from transformers import WhisperForConditionalGeneration, WhisperProcessor
from peft import PeftModel, PeftConfig

class Config:
    base_model_id = "openai/whisper-large-v3"
    lora_model_path = "/kaggle/input/models/leo17messi/nepalienglish-codemix-model/pytorch/default/1/outputs/best_checkpoint"
    csv_path = "/kaggle/input/datasets/panditaadarsh/codeswitchv3/metadata_cycle2.csv"
    audio_dir = "/kaggle/input/datasets/panditaadarsh/nepali-english-codeswitched/kaggle_upload/audios_segment"
    
    test_set_size = 300
    batch_size = 8
    max_generation_length = 225

config = Config()
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")


In [ ]:
import re
import unicodedata
import jiwer
from collections import Counter
from tqdm import tqdm
import librosa
import time

DEVANAGARI_START = 0x0900
DEVANAGARI_END = 0x097F

def asr_normalize(text: str) -> str:
    text = unicodedata.normalize('NFC', text)
    text = text.lower()
    text = re.sub(r'[^\w\s]', '', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

def is_nepali(word: str) -> bool:
    return any(DEVANAGARI_START <= ord(ch) <= DEVANAGARI_END for ch in word)

def is_english(word: str) -> bool:
    has_latin = any('a' <= ch <= 'z' for ch in word)
    has_devanagari = is_nepali(word)
    return has_latin and not has_devanagari

def evaluate_predictions(references, predictions):
    norm_refs = [asr_normalize(r) for r in references]
    norm_preds = [asr_normalize(p) for p in predictions]
    
    out_wer = jiwer.process_words(norm_refs, norm_preds)
    overall_wer = out_wer.wer * 100
    
    out_cer = jiwer.process_characters(norm_refs, norm_preds)
    overall_cer = out_cer.cer * 100
    
    total_nep_s, total_nep_d, total_nep_i, total_nep_n = 0, 0, 0, 0
    total_en_s,  total_en_d,  total_en_i,  total_en_n = 0, 0, 0, 0
    
    for r, p in zip(norm_refs, norm_preds):
        r_nep = ' '.join(w for w in r.split() if is_nepali(w))
        p_nep = ' '.join(w for w in p.split() if is_nepali(w))
        
        if len(r_nep) == 0 and len(p_nep) > 0:
            total_nep_i += len(p_nep.split())
        elif len(r_nep) > 0:
            out_n = jiwer.process_words(r_nep, p_nep)
            total_nep_s += out_n.substitutions
            total_nep_d += out_n.deletions
            total_nep_i += out_n.insertions
            total_nep_n += len(r_nep.split())
            
        r_en = ' '.join(w for w in r.split() if is_english(w))
        p_en = ' '.join(w for w in p.split() if is_english(w))
        
        if len(r_en) == 0 and len(p_en) > 0:
            total_en_i += len(p_en.split())
        elif len(r_en) > 0:
            out_e = jiwer.process_words(r_en, p_en)
            total_en_s += out_e.substitutions
            total_en_d += out_e.deletions
            total_en_i += out_e.insertions
            total_en_n += len(r_en.split())
            
    nep_wer = ((total_nep_s + total_nep_d + total_nep_i) / total_nep_n) * 100 if total_nep_n > 0 else 0
    cm_wer = ((total_en_s + total_en_d + total_en_i) / total_en_n) * 100 if total_en_n > 0 else 0
    
    return {
        "WER": overall_wer,
        "CER": overall_cer,
        "Nep": nep_wer,
        "CM-WER": cm_wer
    }

normalize_text = asr_normalize


In [ ]:
import csv
print('Loading hold-out test set...')
paths, texts = [], []
with open(config.csv_path, 'r', encoding='utf-8') as f:
    reader = csv.reader(f)
    header = next(reader)
    for row in reader:
        if len(row) > 1:
            paths.append(row[0].strip())
            texts.append(','.join(row[1:]).strip())

df = pd.DataFrame({'audio_path': paths, 'text': texts})
df['text'] = df['text'].apply(normalize_text)
df = df[df['text'] != ''].reset_index(drop=True)
df['audio_path'] = df['audio_path'].apply(
    lambda x: x if os.path.isabs(x) else os.path.join(config.audio_dir, os.path.basename(x))
)

mask = df['audio_path'].apply(os.path.exists)
df = df[mask].reset_index(drop=True)

test_df = df.iloc[-config.test_set_size:].reset_index(drop=True)
print(f'Test set ready: {len(test_df)} utterances.')


In [ ]:
def run_inference(model, processor, df, force_lang=None):
    predictions = []
    references = df['text'].tolist()
    audio_paths = df['audio_path'].tolist()
    
    generate_kwargs = {'max_new_tokens': config.max_generation_length}
    if force_lang == 'ne':
        forced_ids = processor.get_decoder_prompt_ids(language="ne", task="transcribe")
        generate_kwargs['forced_decoder_ids'] = forced_ids
    elif force_lang == 'en':
        forced_ids = processor.get_decoder_prompt_ids(language="en", task="transcribe")
        generate_kwargs['forced_decoder_ids'] = forced_ids
    else:
        forced_ids = processor.get_decoder_prompt_ids(task="transcribe")
        generate_kwargs['forced_decoder_ids'] = forced_ids

    for i in tqdm(range(0, len(audio_paths), config.batch_size), desc=f'Inference (Lang={force_lang})'):
        batch_paths = audio_paths[i:i + config.batch_size]
        batch_arrays = []
        for path in batch_paths:
            arr, sr = librosa.load(path, sr=16000, mono=True)
            batch_arrays.append(arr)

        inputs = processor(batch_arrays, sampling_rate=16000, return_tensors='pt', padding=True).to(device)
        if device.type == 'cuda':
            inputs['input_features'] = inputs['input_features'].half()

        with torch.no_grad():
            gen_ids = model.generate(inputs['input_features'], **generate_kwargs)

        preds = processor.batch_decode(gen_ids, skip_special_tokens=True)
        predictions.extend([normalize_text(p) for p in preds])

    return predictions, references


## Run Experiments (Zero-shot vs Fine-Tuned)

In [ ]:
# 1. Load Zero-Shot Baseline Model
print("Loading Zero-Shot Baseline...")
processor = WhisperProcessor.from_pretrained(config.base_model_id)
baseline_model = WhisperForConditionalGeneration.from_pretrained(config.base_model_id).to(device)
if device.type == 'cuda': baseline_model = baseline_model.half()
baseline_model.eval()

# Run Zero-Shot Baseline (Constrained to Nepali)
base_preds, refs = run_inference(baseline_model, processor, test_df, force_lang='ne')
base_metrics = evaluate_predictions(refs, base_preds)

# Run Zero-Shot Baseline (Constrained to English) - TASK 6
base_en_preds, _ = run_inference(baseline_model, processor, test_df, force_lang='en')
base_en_metrics = evaluate_predictions(refs, base_en_preds)

# Clean up baseline to save memory
import gc
del baseline_model
gc.collect()
torch.cuda.empty_cache()

# 2. Load Fine-Tuned LoRA Model
print("\nLoading Fine-Tuned LoRA Model...")
try:
    ft_base = WhisperForConditionalGeneration.from_pretrained(config.base_model_id).to(device)
    ft_model = PeftModel.from_pretrained(ft_base, config.lora_model_path)
    ft_model = ft_model.merge_and_unload()
except ValueError:
    print("adapter_config.json not found, loading as a fully merged model instead.")
    ft_model = WhisperForConditionalGeneration.from_pretrained(config.lora_model_path).to(device)
if device.type == 'cuda': ft_model = ft_model.half()
ft_model.eval()

# Run Fine-Tuned Unconstrained
ft_uncon_preds, _ = run_inference(ft_model, processor, test_df, force_lang=None)
ft_uncon_metrics = evaluate_predictions(refs, ft_uncon_preds)

# Run Fine-Tuned Constrained (Nepali)
ft_con_preds, _ = run_inference(ft_model, processor, test_df, force_lang='ne')
ft_con_metrics = evaluate_predictions(refs, ft_con_preds)


## Verify Byte-Identical Claim

In [ ]:
# Verify Ablation Claim: "Constrained and unconstrained decoding yield byte-identical outputs"
identical_count = sum(1 for u, c in zip(ft_uncon_preds, ft_con_preds) if u == c)
match_percentage = (identical_count / len(refs)) * 100

print("="*60)
print(f"ABLATION RESULT: {match_percentage:.2f}% byte-identical outputs on test set")
print(f"({identical_count} out of {len(refs)} utterances match perfectly)")
print("="*60)


## Table 4 Results

In [ ]:
# Generate Table 4 Data
results_table = [
    {"Model": "Whisper Zero-Shot (Force NE)", **base_metrics},
    {"Model": "Whisper Zero-Shot (Force EN)", **base_en_metrics},
    {"Model": "Whisper-CS (Unconstrained)", **ft_uncon_metrics},
    {"Model": "Whisper-CS (Force NE)", **ft_con_metrics},
]

df_res = pd.DataFrame(results_table).round(2)
print("\nTable 4: Evaluation results on the 300-utterance test set (%)")
display(df_res)


## Paper Metrics: Hallucination, SDI Breakdown, and CMI Correlation

In [ ]:
# Section 5.1: Hallucination Rate
# Count utterances where hypothesis is >2x reference length
hallucination_count = 0
for ref, hyp in zip(refs, ft_uncon_preds):
    ref_len = len(ref.split())
    hyp_len = len(hyp.split())
    if ref_len > 0 and hyp_len > 2 * ref_len:
        hallucination_count += 1

print(f"Hallucination Rate (LoRA Unconstrained): {hallucination_count} out of {len(refs)} utterances ({(hallucination_count/len(refs))*100:.2f}%)")


In [ ]:
# Section 5.4: SDI Error Analysis
total_s, total_d, total_i = 0, 0, 0
for ref, hyp in zip(refs, ft_uncon_preds):
    try:
        out = jiwer.process_words(ref, hyp)
        total_s += out.substitutions
        total_d += out.deletions
        total_i += out.insertions
    except:
        pass

total_errors = total_s + total_d + total_i
if total_errors > 0:
    print(f"Total Errors: {total_errors}")
    print(f"Substitutions: {total_s} ({(total_s/total_errors)*100:.1f}%)")
    print(f"Insertions: {total_i} ({(total_i/total_errors)*100:.1f}%)")
    print(f"Deletions: {total_d} ({(total_d/total_errors)*100:.1f}%)")


In [ ]:
# Section 5.5: CMI vs WER Correlation
from scipy import stats

def compute_cmi(sentence):
    tokens = sentence.split()
    ne_count = sum(1 for t in tokens if is_nepali(t))
    en_count = sum(1 for t in tokens if is_english(t))
    total = ne_count + en_count
    if total == 0: return 0
    return 100 * (1 - max(ne_count, en_count) / total)

cmis, wers = [], []
for ref, hyp in zip(refs, ft_uncon_preds):
    if not ref.strip(): continue
    cmi = compute_cmi(ref)
    if cmi > 0:
        cmis.append(cmi)
        wers.append(jiwer.wer(ref, hyp) * 100)

if len(cmis) > 2:
    r, p = stats.pearsonr(cmis, wers)
    print(f"CMI vs WER Pearson Correlation: r = {r:.3f}, p = {p:.3f}")
else:
    print("Not enough code-mixed sentences to compute correlation.")


In [ ]:
# 3. Evaluate Whisper-Small-CS
print("\nLoading Fine-Tuned Whisper-Small-CS...")
small_base_id = "openai/whisper-small"
small_lora_path = "/kaggle/input/models/panditaadarsh/nepalienglish-codemix-model-small/pytorch/default/1/outputs/best_checkpoint"

processor_small = WhisperProcessor.from_pretrained(small_base_id)
try:
    ft_base_small = WhisperForConditionalGeneration.from_pretrained(small_base_id).to(device)
    ft_model_small = PeftModel.from_pretrained(ft_base_small, small_lora_path)
    ft_model_small = ft_model_small.merge_and_unload()
except ValueError:
    print("adapter_config.json not found, loading as a fully merged model instead.")
    ft_model_small = WhisperForConditionalGeneration.from_pretrained(small_lora_path).to(device)
if device.type == "cuda": ft_model_small = ft_model_small.half()
ft_model_small.eval()

# Unconstrained
ft_small_uncon_preds, _ = run_inference(ft_model_small, processor_small, test_df, force_lang=None)
ft_small_uncon_metrics = evaluate_predictions(refs, ft_small_uncon_preds)

# Constrained
ft_small_con_preds, _ = run_inference(ft_model_small, processor_small, test_df, force_lang="ne")
ft_small_con_metrics = evaluate_predictions(refs, ft_small_con_preds)

import gc
del ft_model_small, ft_base_small
gc.collect()
torch.cuda.empty_cache()


In [ ]:
import json
import base64
import zlib

final_results = {
    "Whisper_Zero_Shot_Force_NE": base_metrics,
    "Whisper_Zero_Shot_Force_EN": base_en_metrics,
    "Whisper_Large_CS_Unconstrained": ft_uncon_metrics,
    "Whisper_Large_CS_Constrained": ft_con_metrics,
    "Whisper_Small_CS_Unconstrained": ft_small_uncon_metrics,
    "Whisper_Small_CS_Constrained": ft_small_con_metrics,
}

print("\n================ SUMMARY OF ALL METRICS ================")
for model_name, metrics in final_results.items():
    print(f"{model_name}:")
    print(f"  WER:    {metrics['WER']:.2f}%")
    print(f"  CER:    {metrics['CER']:.2f}%")
    print(f"  Nep:    {metrics['Nep']:.2f}%")
    print(f"  CM-WER: {metrics['CM-WER']:.2f}%")
    print()

# Create zipped string
json_str = json.dumps(final_results)
compressed = zlib.compress(json_str.encode("utf-8"))
b64_str = base64.b64encode(compressed).decode("utf-8")

print("\n--- COPY THIS ENTIRE STRING AND PASTE IT TO THE ASSISTANT ---")
print(f"ZIPPED_RESULTS_BEGIN:{b64_str}:ZIPPED_RESULTS_END")
